# Track E — Deployable Label Reader Experiment

Run **after** `00_bootstrap.ipynb` has succeeded in this Colab session (tunnel up, GCS mounted, repo cloned at `/content/AutoLearnMeds`).

Drives the full Track E workflow:
1. Pull latest from `phase-0-plumbing`
2. Smoke-test the Track E setup
3. Calibrate Qwen2-VL on val (cheap, decides whether pseudo path is viable)
4. **Run #1** — gold-only-control SigLIP-large-384 (~5h GPU)
5. If calibration passed → full pseudo-labeling pass (~3h GPU)
6. Build the gold+pseudo tokenizer
7. **Run #3** — full Track E with 3-stage schedule (~10h GPU)

Each code cell is self-contained — re-runnable after a Colab disconnect.

**Decision gates** (after each major step):
- After Run #1: safety-4 macro_edit_f1 must be ≥ **0.29** (Track A's 0.2396 + 0.05). Below that → pivot to Approach 2 (OCR-conditioned).
- After Qwen2-VL calibration: safety-4 macro_edit_f1 must be ≥ **0.30**. Below that → skip pseudo path entirely.
- After Run #3: ≥ Run #1 + 0.03 → continue to ablations.


## 0. Pull latest


In [ ]:
%cd /content/AutoLearnMeds
!git fetch origin
!git checkout phase-0-plumbing
!git pull origin phase-0-plumbing
!git log --oneline -5


## 1. Smoke-test the Track E setup (~10 sec)


In [ ]:
!pytest tests/test_track_e_smoke.py tests/test_train_config_compat.py tests/test_train_stage_schedule.py -v


## 2. Qwen2-VL calibration on val (~30 min GPU)

Decides whether to spend ~3h on full pseudo-labeling.

**Gate:** safety-4 macro_edit_f1 ≥ **0.30** → run full labeling. Below that → skip pseudo path; Run #1 alone is the headline.


In [ ]:
!python scripts/codex/pseudo_label_qwen2vl.py \
    --mode calibrate \
    --gold-jsonl raw/val_reconstructed.jsonl \
    --out-jsonl data/pseudo_labels/round_001.jsonl


In [ ]:
# Inspect calibration
import json
from pathlib import Path
cal = Path('data/pseudo_labels/round_001.calibration.json')
if cal.exists():
    print(json.dumps(json.loads(cal.read_text()), indent=2))
else:
    print('No calibration file found — check the calibrate step output.')


## 3. Run #1 — gold-only-control (~5h GPU)

Isolates the resolution-upgrade effect (SigLIP-large-384 vs Track A's SigLIP-base-224). Independent of pseudo-labels.

Survives Colab disconnect via the resume-from-checkpoint logic — if the kernel dies, just re-run this cell and it picks up from the last saved step.


In [ ]:
import subprocess
proc = subprocess.run([
    'bash', 'scripts/run_experiment.sh',
    'track_e_highres_gold_only-seed44',
    '--track', 'E',
    '--config', 'experiments/configs/track_e_highres_gold_only.yaml',
    '--seed', '44',
], capture_output=False)
print(f'Exit code: {proc.returncode}')


In [ ]:
# Finalize Run #1 — append ledger row, regen leaderboard, write plots
!bash scripts/finalize_experiment.sh track_e_highres_gold_only-seed44 --phase confirm


In [ ]:
# Inspect Run #1 metrics
import json
from pathlib import Path
m = Path('experiments/runs/track_e_highres_gold_only-seed44/metrics.json')
if m.exists():
    print(json.dumps(json.loads(m.read_text()), indent=2))
else:
    print('Run not finalized yet.')


## 4. Full pseudo-labeling pass (~3h GPU)

**ONLY run this if Qwen2-VL's val calibration was ≥ 0.30 lenient F1 on safety-4.**

Resume-aware — if interrupted, re-run the same cell; the script picks up from the last processed image via `round_001.progress.jsonl`.


In [ ]:
# Build the unlabeled image list (all images in raw/ minus labeled ones)
import json, subprocess
from pathlib import Path

labeled = set()
with open('raw/golden_set/gold_standard.jsonl') as f:
    for line in f:
        labeled.add(json.loads(line)['image_path'].split('/')[-1])

out = subprocess.check_output(['gsutil', 'ls', 'gs://auto_learn_meds/raw/raw_images/']).decode().splitlines()
unlabeled = [p for p in out if p.lower().endswith(('.jpg', '.png', '.jpeg')) and p.split('/')[-1] not in labeled]

target = Path('data/pseudo_labels/unlabeled_paths.txt')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('\n'.join(unlabeled))
print(f'{len(unlabeled)} unlabeled images → {target}')


In [ ]:
!python scripts/codex/pseudo_label_qwen2vl.py \
    --mode label \
    --unlabeled-list data/pseudo_labels/unlabeled_paths.txt \
    --out-jsonl data/pseudo_labels/round_001.jsonl \
    --resume


## 5. Build the Track E tokenizer (gold + pseudo, ~1 min CPU)


In [ ]:
!python scripts/build_track_e_tokenizer.py \
    --train-jsonl data/processed/train.jsonl \
    --pseudo-jsonl data/pseudo_labels/round_001.jsonl \
    --out experiments/tokenizers/track_e_bpe.json


## 6. Run #3 — full Track E with pseudo + 3-stage (~10h GPU)

3-stage schedule: pseudo warmup (500 steps) → joint (2000 steps) → gold-only finetune (500 steps, LR ×0.1).

Resume-aware on Colab disconnect.


In [ ]:
import subprocess
proc = subprocess.run([
    'bash', 'scripts/run_experiment.sh',
    'track_e_highres_full-seed44',
    '--track', 'E',
    '--config', 'experiments/configs/track_e_highres.yaml',
    '--seed', '44',
], capture_output=False)
print(f'Exit code: {proc.returncode}')


In [ ]:
!bash scripts/finalize_experiment.sh track_e_highres_full-seed44 --phase confirm


In [ ]:
import json
from pathlib import Path
m = Path('experiments/runs/track_e_highres_full-seed44/metrics.json')
if m.exists():
    print(json.dumps(json.loads(m.read_text()), indent=2))


## 7. Compare across tracks

Reads `experiments/safety4_baseline.json` (existing tracks) + the new Track E rows.


In [ ]:
import json
from pathlib import Path
baseline = json.loads(Path('experiments/safety4_baseline.json').read_text())
print(f"{'track':<50s}  {'safety4_macro_edit_f1':>22s}")
for k, v in sorted(baseline.items(), key=lambda x: -x[1]['safety4_macro_edit_f1']):
    print(f"{k:<50s}  {v['safety4_macro_edit_f1']:>22.4f}")

# Track E runs (if they finished)
for run in ['track_e_highres_gold_only-seed44', 'track_e_highres_full-seed44']:
    pf = Path(f'experiments/per_field_{run}.json')
    if pf.exists():
        d = json.loads(pf.read_text())
        sfields = ['batch_number', 'mfg_date', 'expiry_date', 'mrp']
        edit = [d.get('per_field_edit_f1', {}).get(f, 0.0) for f in sfields]
        macro = sum(edit) / len(edit)
        print(f"{run:<50s}  {macro:>22.4f}")


## 8. Next steps after Run #3

**If safety-4 macro_edit_f1 ≥ 0.32** (= Run #1 + 0.03): proceed to ablations in `experiments/configs/`:
- `track_e_highres_full_w05-seed44.yaml` (pseudo_weight=0.5)
- `track_e_highres_full_w10-seed44.yaml` (pseudo_weight=1.0)
- `track_e_highres_full_no_stage3-seed44.yaml` (drop stage 3)
- `track_e_512_full-seed44.yaml` (resolution 512)
- `track_e_dinov2_full-seed44.yaml` (DINOv2 encoder)

(These configs aren't yet created; spin them up by copying `track_e_highres.yaml` and changing one knob.)

**If under that threshold:** stop the pseudo path. The best of Run #1 or Run #3 is the candidate for the test-set claim.

**Test-set claim run** (touches the held-out test split, used ONCE — for the paper):

```python
# Eval the best checkpoint on test.jsonl
from prepare import evaluate_test
metrics = evaluate_test(model, 'data/processed/test.jsonl', images_root='/mnt/gcs/raw/raw_images')
```
